# pyscf stub typing on Colab (CPU)

Reproduces the local `gen_pyscf_stubs.py` pipeline end-to-end on a **CPU** Colab
runtime - same two stages, same scripts (embedded verbatim below):

1. **TRACE** - run each `pyscf/**/test/` suite under **MonkeyType**
   (`monkeytype run -m pytest ...`), recording observed arg/return types into
   `monkeytype_pyscf.sqlite3`. Failing numerical asserts still execute, so they
   still get traced.
2. **MERGE** - index the trace DB, then overlay observed types onto the
   `stubgen` skeletons in `pyscf-stubs/` with libcst. Ambiguous duplicate PySCF
   class names (`RHF`, `HF1e`, `GKS`, ...) are degraded to `Incomplete` instead
   of dropping whole modules. A known MonkeyType/libcst embedded-import rendering
   issue is repaired before parsing.

**Environment differences from local:** we `pip install` a pinned pyscf wheel,
then `git clone` the *matching* tagged source only to copy its test suites into
the installed package (pyscf wheels don't ship tests). The working dir `ROOT` is
Python's `site-packages`, so `ROOT/pyscf` is the real compiled package and
MonkeyType's file filter stays consistent.

**Runtime:** no GPU needed. The run cell defaults to **all** subpackages
(`SUBPKGS = ""`), which is a multi-hour CPU run even on Colab Pro. Set `SUBPKGS`
to a space-separated subset if you want a smaller fresh run. **Run the cells
top-to-bottom.**


## 1. Install dependencies

In [ ]:
PYSCF_VERSION = "2.13.1"   # pinned so the cloned test suite matches the wheel
!pip install -q "pyscf=={PYSCF_VERSION}" monkeytype libcst tqdm mypy
print("installed pyscf", PYSCF_VERSION, "+ monkeytype, libcst, tqdm, mypy (stubgen)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.4/518.4 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.8 MB/s eta 0:00:00
installed pyscf 2.13.1 + monkeytype, libcst, tqdm, mypy (stubgen)


## 2. Locate the installed pyscf and set `ROOT`

`ROOT = site-packages`, so `ROOT/pyscf` is the compiled package. We `chdir` here
so the embedded scripts (which derive `ROOT` from their own location) and
`%%writefile` all agree on the same directory.

In [ ]:
import os, sys, pyscf
from pathlib import Path

PKG  = Path(pyscf.__file__).resolve().parent   # .../site-packages/pyscf
ROOT = PKG.parent                              # .../site-packages
os.chdir(ROOT)
print("pyscf :", pyscf.__version__)
print("PKG   :", PKG)
print("ROOT  :", ROOT, "(cwd)")

pyscf : 2.13.1
PKG   : /usr/local/lib/python3.12/dist-packages/pyscf
ROOT  : /usr/local/lib/python3.12/dist-packages (cwd)


## 3. Write the pipeline scripts into `ROOT`
These are the exact local files, unmodified.

In [ ]:
%%writefile monkeytype_config.py
"""MonkeyType configuration for generating pyscf type stubs.

Discovered automatically by the ``monkeytype`` CLI when this module is on
PYTHONPATH (run the CLI from the checkout root).

- Traces are restricted to the pyscf package itself so we don't record
  types for numpy/scipy/pytest internals.
- Traces accumulate in ``monkeytype_pyscf.sqlite3`` so multiple test-suite
  runs (one per subpackage) build up a single store.
"""

import os

from monkeytype.config import DefaultConfig
from monkeytype.db.sqlite import SQLiteStore

import pyscf

# Absolute path to the installed pyscf package (the local checkout).
PYSCF_DIR = os.path.dirname(os.path.abspath(pyscf.__file__))

TRACE_STORE = os.path.join(os.path.dirname(__file__), "monkeytype_pyscf.sqlite3")


def _pyscf_only(code):
    """Only trace functions defined inside the pyscf package."""
    filename = code.co_filename
    return bool(filename) and filename.startswith(PYSCF_DIR)


class PyscfConfig(DefaultConfig):
    def trace_store(self):
        return SQLiteStore.make_store(TRACE_STORE)

    def code_filter(self):
        return _pyscf_only


CONFIG = PyscfConfig()


Writing monkeytype_config.py


In [ ]:
%%writefile merge_traced_stubs.py
#!/usr/bin/env python
"""Merge MonkeyType-observed types into the stubgen skeleton stubs.

For each module recorded in the MonkeyType trace store:
  * skeleton  = pyscf-stubs/<path>.pyi  (full structure, mostly `Incomplete`)
  * typed     = `monkeytype stub <module>` output  (rich observed types, but
                only for functions actually exercised by the test suite)

We use libcst's ApplyTypeAnnotationsVisitor (the same merge engine behind
`monkeytype apply`) to overlay the observed annotations onto the skeleton,
so the result keeps every member stubgen found *and* gains real types where
the tests observed them.

Usage:
    python merge_traced_stubs.py [module ...]   # default: all traced modules
"""

import re
import sqlite3
import subprocess
import sys
from pathlib import Path

import libcst as cst
from libcst.codemod import CodemodContext
from libcst.codemod.visitors import ApplyTypeAnnotationsVisitor

ROOT = Path(__file__).resolve().parent
STUBS_ROOT = ROOT / "pyscf-stubs"
TRACE_STORE = ROOT / "monkeytype_pyscf.sqlite3"

# Resolve the monkeytype executable next to the running interpreter so this
# works whether or not pyscfenv is "activated" on PATH.
_MT = Path(sys.executable).parent / "monkeytype"
MONKEYTYPE = str(_MT) if _MT.exists() else "monkeytype"


def ensure_trace_store_index() -> None:
    """Index the trace DB for fast per-module MonkeyType stub generation."""
    if not TRACE_STORE.exists():
        return
    with sqlite3.connect(TRACE_STORE) as con:
        con.execute(
            "CREATE INDEX IF NOT EXISTS monkeytype_call_traces_module_idx "
            "ON monkeytype_call_traces(module)"
        )
        con.execute("ANALYZE monkeytype_call_traces")


def traced_modules():
    out = subprocess.run(
        [MONKEYTYPE, "list-modules"], capture_output=True, text=True, cwd=ROOT
    )
    return [m for m in out.stdout.split() if m.startswith("pyscf")]


def skeleton_path(module: str) -> Path:
    parts = module.split(".")
    pkg_init = STUBS_ROOT.joinpath(*parts, "__init__.pyi")
    if pkg_init.exists():
        return pkg_init
    return STUBS_ROOT.joinpath(*parts).with_suffix(".pyi")


def monkeytype_stub(module: str) -> str | None:
    res = subprocess.run(
        [MONKEYTYPE, "stub", module], capture_output=True, text=True, cwd=ROOT
    )
    if res.returncode != 0 or not res.stdout.strip():
        return None
    return res.stdout


def _merge_once(skeleton_src: str, typed_src: str) -> str:
    context = CodemodContext()
    ApplyTypeAnnotationsVisitor.store_stub_in_context(
        context,
        cst.parse_module(typed_src),
        overwrite_existing_annotations=True,
    )
    visitor = ApplyTypeAnnotationsVisitor(context)
    result = visitor.transform_module(cst.parse_module(skeleton_src))
    return result.code


# libcst raises this when a name in an annotation resolves to >1 qualified name.
_AMBIGUOUS_RE = re.compile(
    r"Could not resolve a unique qualified name for type (\S+) at"
)
_IMPORT_EXPR_RE = re.compile(
    r"\bfrom\s+([A-Za-z_]\w*(?:\.[A-Za-z_]\w*)*)\s+import\s+([A-Za-z_]\w*)\b"
)


def _repair_embedded_import_annotations(typed_src: str, repairs=None) -> str:
    """Repair MonkeyType output that embeds import statements as annotations.

    Occasionally MonkeyType/libcst can render an observed type as an expression
    fragment such as ``from numpy import ndarray`` inside a function annotation.
    That is invalid syntax there, but the intended annotation is just
    ``ndarray`` plus the corresponding import.
    """
    imports = set()
    changed = False
    lines = []
    for line in typed_src.splitlines(keepends=True):
        stripped = line.lstrip()
        if stripped.startswith(("from ", "import ")):
            lines.append(line)
            continue

        def replace(match):
            nonlocal changed
            changed = True
            imports.add((match.group(1), match.group(2)))
            return match.group(2)

        lines.append(_IMPORT_EXPR_RE.sub(replace, line))

    if not changed:
        return typed_src

    src = "".join(lines)
    for module, name in sorted(imports):
        stmt = f"from {module} import {name}\n"
        if stmt not in src:
            src = stmt + src
    if repairs is not None:
        repairs.extend(f"{module}.{name}" for module, name in sorted(imports))
    return src


class _NeutralizeAnnotationNames(cst.CSTTransformer):
    """Rewrite the given bare names to ``Incomplete`` where they appear inside
    an annotation (leaving imports/other code untouched)."""

    def __init__(self, names):
        self.names = set(names)
        self._in_annotation = 0

    def visit_Annotation(self, node: cst.Annotation) -> None:
        self._in_annotation += 1

    def leave_Annotation(self, original_node, updated_node):
        self._in_annotation -= 1
        return updated_node

    def leave_Name(self, original_node, updated_node):
        if self._in_annotation and updated_node.value in self.names:
            return updated_node.with_changes(value="Incomplete")
        return updated_node


def _neutralize(typed_src: str, names) -> str:
    src = cst.parse_module(typed_src).visit(
        _NeutralizeAnnotationNames(names)
    ).code
    if "Incomplete" in src and "import Incomplete" not in src:
        src = "from _typeshed import Incomplete\n" + src
    return src


def merge(skeleton_src: str, typed_src: str, *, dropped=None, repairs=None,
          max_passes: int = 50) -> str:
    """Overlay MonkeyType-observed annotations onto the stubgen skeleton.

    pyscf reuses class short-names (RHF, HF1e, GKS, ...) across modules, so a
    MonkeyType stub can reference a name libcst cannot resolve to a single
    qualified name, which otherwise aborts the whole module. Instead we degrade
    just those ambiguous annotations to ``Incomplete`` and retry, so every other
    observed type is still merged. Names degraded this way are appended to the
    optional ``dropped`` list for reporting.
    """
    typed = _repair_embedded_import_annotations(typed_src, repairs=repairs)
    for _ in range(max_passes):
        try:
            return _merge_once(skeleton_src, typed)
        except ValueError as exc:
            m = _AMBIGUOUS_RE.search(str(exc))
            # Only auto-handle bare (non-dotted) ambiguous names; re-raise the rest.
            if not m or "." in m.group(1):
                raise
            bad = m.group(1)
            new_typed = _neutralize(typed, [bad])
            if new_typed == typed:
                raise  # nothing changed -> avoid an infinite loop
            typed = new_typed
            if dropped is not None:
                dropped.append(bad)
    raise ValueError("merge did not converge (too many ambiguous names)")


def main(argv):
    ensure_trace_store_index()
    modules = argv or traced_modules()
    merged = skipped = failed = 0
    for module in modules:
        skel = skeleton_path(module)
        if not skel.exists():
            skipped += 1  # e.g. test modules, which stubgen did not stub
            continue
        typed_src = monkeytype_stub(module)
        if typed_src is None:
            skipped += 1
            continue
        repairs = []
        try:
            out = merge(skel.read_text(), typed_src, repairs=repairs)
        except Exception as exc:  # noqa: BLE001 - report and continue
            failed += 1
            print(f"  FAIL {module}: {type(exc).__name__}: {exc}", file=sys.stderr)
            continue
        skel.write_text(out)
        merged += 1
        print(f"  merged {module}")
        if repairs:
            names = ", ".join(sorted(set(repairs)))
            print(f"    repaired embedded import annotations: {names}")
    print(f"\nmerged={merged} skipped={skipped} failed={failed}")


if __name__ == "__main__":
    main(sys.argv[1:])


Writing merge_traced_stubs.py


In [ ]:
%%writefile gen_pyscf_stubs.py
#!/usr/bin/env python
"""End-to-end pyscf stub typing: MonkeyType tracing -> merge into pyscf-stubs/.

Runs the whole pipeline globally across pyscf:

  1. TRACE  - discover every ``pyscf/**/test[s]/`` suite, run each under
              MonkeyType (``monkeytype run -m pytest ...``), accumulating
              observed argument/return types into monkeytype_pyscf.sqlite3.
              Test failures are tolerated - a failing numerical assertion
              still executes (and therefore traces) the function.
  2. MERGE  - for every traced module, overlay the observed types onto the
              stubgen skeleton in pyscf-stubs/ via libcst (see
              merge_traced_stubs.py). Structure/constants are preserved;
              only annotations are filled in.

Assumes the stubgen skeletons already exist under pyscf-stubs/ (created with
``stubgen -p pyscf -o pyscf-stubs``). Run this from the checkout root with the
pyscfenv interpreter, e.g.:

    ~/miniconda3/envs/pyscfenv/bin/python gen_pyscf_stubs.py

Examples:
    python gen_pyscf_stubs.py                 # trace ALL suites, then merge
    python gen_pyscf_stubs.py --only scf dft  # limit to these subpackages
    python gen_pyscf_stubs.py --skip pbc      # trace everything except pbc
    python gen_pyscf_stubs.py --list          # show discovered suites, exit
    python gen_pyscf_stubs.py --merge-only    # skip tracing, just re-merge
    python gen_pyscf_stubs.py --trace-only    # trace, don't merge
"""

import argparse
import os
import subprocess
import sys
import threading
import time
from pathlib import Path

try:
    from tqdm import tqdm
    HAVE_TQDM = True
except ImportError:  # graceful fallback: plain prints, no bar
    HAVE_TQDM = False

# Reuse the proven merge primitives.
from merge_traced_stubs import (
    TRACE_STORE,
    ensure_trace_store_index,
    merge,
    monkeytype_stub,
    skeleton_path,
    traced_modules,
)

ROOT = Path(__file__).resolve().parent
PKG_DIR = ROOT / "pyscf"
STUBS_ROOT = ROOT / "pyscf-stubs"

# Resolve the monkeytype executable next to the running interpreter so the
# script works whether or not pyscfenv is "activated" on PATH.
BIN_DIR = Path(sys.executable).parent
MONKEYTYPE = str(BIN_DIR / "monkeytype")
if not Path(MONKEYTYPE).exists():
    MONKEYTYPE = "monkeytype"


def subpackage_of_suite(suite: Path) -> str:
    """Top-level subpackage name for a test dir, e.g. pyscf/gto/test -> 'gto'."""
    return suite.relative_to(PKG_DIR).parts[0]


def subpackage_of_module(module: str) -> str:
    """Top-level subpackage for a dotted module, e.g. pyscf.gto.mole -> 'gto'."""
    parts = module.split(".")
    return parts[1] if len(parts) > 1 else ""


def is_test_module(module: str) -> bool:
    """True for the test modules themselves (traced, but never stubbed)."""
    parts = module.split(".")
    return "test" in parts or "tests" in parts or parts[-1].startswith("test_")


def discover_suites():
    """All test directories under pyscf, sorted, as (subpackage, path)."""
    suites = []
    for path in sorted(PKG_DIR.rglob("*")):
        if path.is_dir() and path.name in ("test", "tests"):
            suites.append((subpackage_of_suite(path), path))
    return suites


def select(names, only, skip):
    """Filter a set of subpackage names by --only / --skip."""
    if only:
        names = [n for n in names if n in only]
    if skip:
        names = [n for n in names if n not in skip]
    return names


def trace_target(target: Path, timeout: int, env: dict, extra, on_tick=None):
    """Run ``monkeytype run -m pytest <target>`` under a timeout.

    ``target`` may be a whole suite directory or a single test file. ``on_tick``,
    if given, is called ~once/second with the elapsed seconds so a progress bar
    can show a live timer (each suite is a long-running subprocess, so without
    this the bar looks frozen). Returns
    ``(status, elapsed, timed_out, returncode)``.

    NB: MonkeyType only writes traces to the store when its run finishes, so a
    timeout loses that target's traces entirely - prefer --per-file for slow,
    many-file suites (e.g. adc) so one slow file can't discard the rest.
    """
    cmd = [
        MONKEYTYPE, "run", "-m", "pytest",
        str(target), "-q", "-p", "no:cacheprovider", *extra,
    ]
    start = time.time()
    stop = threading.Event()
    if on_tick is not None:
        def _ticker():
            while not stop.wait(1.0):
                on_tick(time.time() - start)
        threading.Thread(target=_ticker, daemon=True).start()
    try:
        res = subprocess.run(
            cmd, cwd=ROOT, env=env, capture_output=True, text=True, timeout=timeout
        )
        timed_out = False
    except subprocess.TimeoutExpired:
        res, timed_out = None, True
    finally:
        stop.set()
    elapsed = time.time() - start
    if timed_out:
        return f"TIMEOUT after {timeout}s", elapsed, True, None
    # Grab pytest's one-line summary (last non-empty line) for context.
    tail = [ln for ln in res.stdout.splitlines() if ln.strip()]
    summary = tail[-1] if tail else f"exit {res.returncode}"
    return f"{summary}  [{elapsed:.0f}s]", elapsed, False, res.returncode


def trace_units(selected, per_file):
    """Expand selected suites into trace units: whole dirs, or per test file."""
    units = []
    for sp, suite in discover_suites():
        if sp not in selected:
            continue
        if per_file:
            files = sorted(suite.rglob("test_*.py"))
            units.extend((sp, f) for f in (files or [suite]))
        else:
            units.append((sp, suite))
    return units


def do_trace(selected, timeout, per_file, extra):
    units = trace_units(selected, per_file)
    if not units:
        print("No test suites match the selection.")
        return
    env = dict(os.environ)
    env["PYTHONPATH"] = os.pathsep.join(
        [str(ROOT), env.get("PYTHONPATH", "")]
    ).rstrip(os.pathsep)
    env["PATH"] = os.pathsep.join([str(BIN_DIR), env.get("PATH", "")])
    kind = "file" if per_file else "suite"
    print(f"== TRACE: {len(units)} {kind}(s), {timeout}s timeout each ==")

    tally = {"ok": 0, "fail": 0, "timeout": 0}

    def classify(returncode, timed_out):
        if timed_out:
            tally["timeout"] += 1
        elif returncode == 0:
            tally["ok"] += 1
        else:
            tally["fail"] += 1

    bar = (tqdm(total=len(units), unit=kind, dynamic_ncols=True)
           if HAVE_TQDM else None)
    for i, (sp, target) in enumerate(units, 1):
        rel = target.relative_to(ROOT)
        if bar is not None:
            bar.set_description(str(rel))
            status, _, timed_out, returncode = trace_target(
                target, timeout, env, extra,
                on_tick=lambda e: bar.set_postfix_str(f"{e:4.0f}s/{timeout}s"),
            )
            classify(returncode, timed_out)
            bar.write(f"  {rel} ... {status}")
            bar.set_postfix_str(
                f"ok={tally['ok']} fail={tally['fail']} timeout={tally['timeout']}"
            )
            bar.update(1)
        else:
            print(f"[{i}/{len(units)}] {rel} ... ", end="", flush=True)
            status, _, timed_out, returncode = trace_target(target, timeout, env, extra)
            classify(returncode, timed_out)
            print(status)
    if bar is not None:
        bar.close()
    print(f"trace done: ok={tally['ok']} fail={tally['fail']} "
          f"timeout={tally['timeout']}")


def prepare_trace_store_for_merge():
    if not TRACE_STORE.exists():
        print(f"trace DB not found at {TRACE_STORE} (did tracing run?)")
        return
    start = time.time()
    print("== INDEX: monkeytype_pyscf.sqlite3 for fast per-module merge ==",
          flush=True)
    ensure_trace_store_index()
    print(f"index ready [{time.time() - start:.0f}s]", flush=True)


def do_merge(selected):
    modules = sorted(
        m for m in traced_modules()
        if subpackage_of_module(m) in selected and not is_test_module(m)
    )
    if not modules:
        print("No traced modules match the selection (did tracing run?).")
        return
    print(f"== MERGE: {len(modules)} traced module(s) into pyscf-stubs/ ==")
    merged = skipped = failed = degraded = repaired = 0
    bar = (tqdm(total=len(modules), unit="module", dynamic_ncols=True)
           if HAVE_TQDM else None)

    def note(message):
        if bar is not None:
            bar.write(message)
        else:
            print(message, flush=True)

    for i, module in enumerate(modules, 1):
        if bar is not None:
            bar.set_description(module)
        elif i == 1 or i % 25 == 0:
            print(f"[{i}/{len(modules)}] merging {module}", flush=True)
        skel = skeleton_path(module)
        if not skel.exists():
            skipped += 1
            if bar is not None:
                bar.update(1)
            continue
        typed_src = monkeytype_stub(module)
        if typed_src is None:
            skipped += 1
            if bar is not None:
                bar.update(1)
            continue
        dropped = []
        repairs = []
        try:
            out = merge(skel.read_text(), typed_src, dropped=dropped,
                        repairs=repairs)
        except Exception as exc:  # noqa: BLE001
            failed += 1
            note(f"  FAIL {module}: {type(exc).__name__}: {exc}")
            if bar is not None:
                bar.update(1)
            continue
        skel.write_text(out)
        merged += 1
        if dropped:
            degraded += 1
            names = ", ".join(sorted(set(dropped)))
            note(f"  ~ {module}: ambiguous {names} -> Incomplete")
        if repairs:
            repaired += 1
            names = ", ".join(sorted(set(repairs)))
            note(f"  ~ {module}: repaired embedded import annotations: {names}")
        if bar is not None:
            bar.set_postfix_str(
                f"merged={merged} degraded={degraded} "
                f"repaired={repaired} skipped={skipped} failed={failed}"
            )
            bar.update(1)
    if bar is not None:
        bar.close()
    print(f"merged={merged} degraded={degraded} repaired={repaired} "
          f"skipped={skipped} failed={failed}")


def main():
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--only", nargs="*", metavar="PKG",
                    help="limit to these top-level subpackages (e.g. scf dft cc)")
    ap.add_argument("--skip", nargs="*", metavar="PKG", default=[],
                    help="exclude these top-level subpackages (e.g. pbc)")
    ap.add_argument("--timeout", type=int, default=1200,
                    help="per-unit timeout in seconds (default 1200)")
    ap.add_argument("--per-file", action="store_true",
                    help="trace each test_*.py separately (finer progress; a "
                         "timeout on one file no longer discards the whole suite)")
    ap.add_argument("--k", metavar="EXPR",
                    help="pytest -k expression, e.g. --k 'not high_cost'")
    ap.add_argument("--trace-only", action="store_true", help="trace, do not merge")
    ap.add_argument("--merge-only", action="store_true", help="skip tracing, only merge")
    ap.add_argument("--list", action="store_true",
                    help="list discovered test suites and exit")
    args = ap.parse_args()

    if not STUBS_ROOT.exists():
        sys.exit(f"error: {STUBS_ROOT} not found - run "
                 f"`stubgen -p pyscf -o pyscf-stubs` first.")

    all_subpkgs = sorted({sp for sp, _ in discover_suites()})
    selected = set(select(all_subpkgs, args.only, args.skip))

    if args.list:
        for sp, suite in discover_suites():
            mark = " " if sp in selected else "-"
            print(f" [{mark}] {suite.relative_to(ROOT)}")
        print(f"\n{len(selected)}/{len(all_subpkgs)} subpackages selected: "
              f"{', '.join(sorted(selected))}")
        return

    if not args.merge_only:
        extra = ["-k", args.k] if args.k else []
        do_trace(selected, args.timeout, args.per_file, extra)
    if not args.trace_only:
        prepare_trace_store_for_merge()
        do_merge(selected)


if __name__ == "__main__":
    main()


Writing gen_pyscf_stubs.py


## 4. Generate the stubgen skeletons

`stubgen -p pyscf -o pyscf-stubs` emits an `Incomplete`‑filled `.pyi` skeleton
for every module. Done *before* copying tests in, so test files don't get
stubbed. Takes a couple of minutes.

**Run from a neutral cwd (`/content`), not `ROOT`.** `ROOT` is `site-packages`,
which contains `mypy_extensions.py`; with cwd there, mypy thinks it *shadows* the
real `mypy_extensions` library and aborts stubgen with a *Critical error during
semantic analysis*, leaving `pyscf-stubs/` empty (every module then shows up as
`skipped` in section 7's merge). Output still goes to `ROOT/pyscf-stubs` via the
absolute `-o` path, so the later steps find it.

In [ ]:
import subprocess
STUBS = ROOT / "pyscf-stubs"

# Run stubgen from a NEUTRAL cwd (NOT ROOT = site-packages). If cwd is
# site-packages, mypy sees site-packages/mypy_extensions.py as a top-level
# module that "shadows" the real mypy_extensions library and aborts with a
# "Critical error during semantic analysis" -> pyscf-stubs comes out EMPTY and
# every module then reports `skipped` at merge time (merged=0 skipped=N).
r = subprocess.run(["stubgen", "-p", "pyscf", "-o", str(STUBS)],
                   cwd="/content", capture_output=True, text=True)
print("\n".join((r.stdout + r.stderr).splitlines()[-5:]))

pyi = list((STUBS / "pyscf").rglob("*.pyi"))
print(f"\nstubgen rc={r.returncode} | {len(pyi)} skeletons under {STUBS / 'pyscf'}")
assert r.returncode == 0 and pyi, "stubgen failed or produced no skeletons - check the error above before continuing"


Generated files under /usr/local/lib/python3.12/dist-packages/pyscf-stubs/pyscf/
/usr/local/lib/python3.12/dist-packages/pyscf/mp/mp2f12_slow.py:37: UserWarning: Module MP2-F12 is under testing
  warnings.warn('Module MP2-F12 is under testing')
/usr/local/lib/python3.12/dist-packages/pyscf/solvent/ddpcm.py:38: UserWarning: Module ddPCM is under testing
  warnings.warn('Module ddPCM is under testing')

stubgen rc=0 | 659 skeletons under /usr/local/lib/python3.12/dist-packages/pyscf-stubs/pyscf


## 5. Fetch the test suites (matching version) into the installed package

pyscf wheels don't ship tests, so clone the matching tag and copy every
`test/`/`tests/` dir into `site-packages/pyscf`. Falls back to the default
branch if the exact tag is missing.

In [ ]:
import shutil, subprocess
SRC = Path("/content/pyscf-src")
if not SRC.exists():
    tag = f"v{PYSCF_VERSION}"
    rc = subprocess.run(["git", "clone", "--depth", "1", "--branch", tag,
                         "https://github.com/pyscf/pyscf", str(SRC)]).returncode
    if rc != 0:
        print(f"tag {tag} not found -> cloning default branch")
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/pyscf/pyscf", str(SRC)], check=True)

src_pkg = SRC / "pyscf"
copied = 0
for d in sorted(src_pkg.rglob("*")):
    if d.is_dir() and d.name in ("test", "tests"):
        dst = PKG / d.relative_to(src_pkg)
        if not dst.exists():
            shutil.copytree(d, dst)
            copied += 1
print(f"copied {copied} test dirs into {PKG}")

copied 53 test dirs into /usr/local/lib/python3.12/dist-packages/pyscf


## 6. Sanity check: discovered suites

In [ ]:
!python gen_pyscf_stubs.py --list | tail -n 15

 [ ] pyscf/pbc/tools/test
 [ ] pyscf/pbc/x2c/test
 [ ] pyscf/qmmm/pbc/test
 [ ] pyscf/qmmm/test
 [ ] pyscf/scf/test
 [ ] pyscf/sgx/grad/test
 [ ] pyscf/sgx/test
 [ ] pyscf/solvent/test
 [ ] pyscf/soscf/test
 [ ] pyscf/symm/test
 [ ] pyscf/tdscf/test
 [ ] pyscf/tools/test
 [ ] pyscf/x2c/test

32/32 subpackages selected: adc, agf2, ao2mo, cc, ci, df, dft, eph, fci, geomopt, grad, gto, gw, hessian, lib, lo, mcpdft, mcscf, md, mp, mrpt, nac, pbc, qmmm, scf, sgx, solvent, soscf, symm, tdscf, tools, x2c


## 7. Run the pipeline (trace -> merge)

`--per-file` traces each `test_*.py` separately so one slow file can't discard a
whole suite (and a timeout no longer loses everything - MonkeyType only flushes
on completion). Before merging, the script indexes `monkeytype_pyscf.sqlite3` so
per-module `monkeytype stub` calls do not repeatedly scan a multi-GB table. Edit
`SUBPKGS`:
- `""` - everything (default; hours on CPU)
- `"gto ao2mo"` - quick demo
- `"gto scf ao2mo lib"` - a broader set


In [ ]:
import shlex, subprocess, sys

SUBPKGS = ""              # space-separated top-level subpackages; "" = ALL
TIMEOUT = 3600            # per-file timeout (s)

cmd = [sys.executable, "gen_pyscf_stubs.py", "--per-file", "--timeout", str(TIMEOUT)]
if SUBPKGS.strip():
    cmd += ["--only", *SUBPKGS.split()]
print(">>", " ".join(shlex.quote(part) for part in cmd), "\n")
r = subprocess.run(cmd, cwd=ROOT)
print("pipeline rc=", r.returncode)
assert r.returncode == 0, "pipeline failed before download; inspect the output above"


>> /usr/bin/python3 gen_pyscf_stubs.py --per-file --timeout 3600 



## 8. Peek at the result
Functions that gained real (non‑`Incomplete`) types from tracing:

In [ ]:
sample = ROOT / "pyscf-stubs/pyscf/gto/mole.pyi"
if sample.exists():
    hits = [ln for ln in sample.read_text().splitlines()
            if "->" in ln and "Incomplete" not in ln]
    print(f"{sample.name}: {len(hits)} typed signatures, e.g.")
    print("\n".join(hits[:30]))
else:
    print("run cell 7 with 'gto' in SUBPKGS first")

mole.pyi: 12 typed signatures, e.g.
def search_ao_r(mol, atm_id, l, j, m, atmshell) -> None: ...
    def __init__(self) -> None: ...
    def nelec(self, neleca_nelecb) -> None: ...
    def nelectron(self, n) -> None: ...
    def multiplicity(self, x) -> None: ...
    def ms(self, x) -> None: ...
    def enuc(self, x) -> None: ...
    def set_range_coulomb(self, omega) -> None: ...
    def set_f12_zeta(self, zeta) -> None: ...
    def nao(self, x) -> None: ...
    def __init__(self, **kwargs) -> None: ...
    def __init__(self, fn, name) -> None: ...


## 9. Download the stubs

In [ ]:
import shutil

stubs_dir = ROOT / "pyscf-stubs"
assert stubs_dir.exists(), "pyscf-stubs does not exist - run sections 4 and 7 first"
pyi = list((stubs_dir / "pyscf").rglob("*.pyi"))
assert pyi, "pyscf-stubs contains no .pyi files"

zip_path = shutil.make_archive("/content/pyscf-stubs", "zip",
                               root_dir=str(ROOT), base_dir="pyscf-stubs")
print(f"zipped {len(pyi)} .pyi files -> {zip_path}")
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("(download works from the Colab UI)", e)


zipped 659 .pyi files -> /content/pyscf-stubs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>